In [ ]:
import pandas as pd
import json
import psycopg2

In [ ]:
# 1. Connect to PostgreSQL database
def connect_to_postgres():
    conn = psycopg2.connect(
        host="localhost",       
        database="lombardia_air_quality", 
        user="airdata_user",    
        password="user"
    )
    return conn

In [ ]:
# 2. Create table if not exists 
def create_table_if_not_exists(conn, table_name, data_sample):
    cursor = conn.cursor()
    
    # Create schema SQL statement based on the data structure
    columns = []
    for key, value in data_sample.items():
        column_type = "TEXT"  # Default type
        
        # Try to infer the data type
        if isinstance(value, int):
            column_type = "INTEGER"
        elif isinstance(value, float):
            column_type = "NUMERIC"
        elif isinstance(value, bool):
            column_type = "BOOLEAN"
        elif isinstance(value, dict):
            column_type = "JSONB"  # Use JSONB for nested structures
        # Special case for timestamp fields
        elif key == "datastart" or key == "datastop":
            column_type = "TIMESTAMP"

        columns.append(f"\"{key}\" {column_type}")

    create_table_sql = f"""
    DROP TABLE IF EXISTS {table_name};
    CREATE TABLE {table_name} (
        id SERIAL PRIMARY KEY,
        {', '.join(columns)}
    );
    """
    
    cursor.execute(create_table_sql)
    conn.commit()
    cursor.close()